<a href="https://colab.research.google.com/github/jeshwanth-A/defi_aiml/blob/main/notebooks/rag_prepare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DefiDoza RAG Preparation Notebook

This notebook is self-contained and only for knowledge-base preparation.

- No local project files required
- Builds a lightweight TF-IDF knowledge index
- Saves artifacts to Google Drive for local assistant use
- Scaffolds starter docs if the Drive knowledge folder is empty

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_BASE = '/content/drive/MyDrive/defidoza'
KNOWLEDGE_DIR = os.path.join(DRIVE_BASE, 'knowledge')
RAG_DIR = os.path.join(DRIVE_BASE, 'rag')
INDEX_PATH = os.path.join(RAG_DIR, 'knowledge_index.pkl')
MANIFEST_PATH = os.path.join(RAG_DIR, 'knowledge_manifest.json')
EMBEDDING_MODEL = 'tfidf'
CHUNK_SIZE = 900
CHUNK_OVERLAP = 150

os.makedirs(KNOWLEDGE_DIR, exist_ok=True)
os.makedirs(RAG_DIR, exist_ok=True)

print('Knowledge dir:', KNOWLEDGE_DIR)
print('RAG dir:', RAG_DIR)

In [ ]:
import json, os, pickle
from datetime import datetime
from pathlib import Path

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

STARTER_DOCS = {
    'project_notes.md': '''# DefiDoza Project Notes

DefiDoza combines live crypto forecasting tools with a curated knowledge base.

Use retrieved docs for explanations, token summaries, risks, and project notes. Use live tools for prices, history, and forecasts. Blend both when the question needs context plus current market data.
''',
    'bitcoin.md': '''# Bitcoin Overview

Bitcoin is the market reference asset in many crypto discussions. It is often framed around scarcity, liquidity, and macro sensitivity.

## Key Risks

- large drawdowns during risk-off periods
- regulatory or custody access changes
- sentiment-led swings even when long-term conviction remains high
''',
    'ethereum.md': '''# Ethereum Overview

Ethereum is the leading smart-contract platform in many DeFi and application narratives.

## Key Risks

- fee spikes
- competitive pressure from other chains
- ecosystem risk from protocols built on top of Ethereum
''',
    'uniswap.md': '''# Uniswap Overview

Uniswap is a decentralized exchange protocol built around automated market making.

## Key Risks

- governance and smart-contract risk
- competitive fee pressure
- protocol usage and UNI token price can diverge
''',
    'glossary.md': '''# Crypto Glossary

## RAG

Retrieval-augmented generation means searching documents at question time and passing the relevant text into the final answer.

## Forecast

A forecast is a model-generated estimate, not a guarantee.
''',
}

def ensure_starter_docs():
    existing = list(Path(KNOWLEDGE_DIR).glob('*.md'))
    if existing:
        print(f'Knowledge folder already has {len(existing)} markdown files.')
        return
    for name, content in STARTER_DOCS.items():
        Path(KNOWLEDGE_DIR, name).write_text(content.strip() + '\n', encoding='utf-8')
    print(f'Wrote {len(STARTER_DOCS)} starter docs to {KNOWLEDGE_DIR}')

def chunk_text(text, chunk_size=900, chunk_overlap=150):
    text = text.replace('\r\n', '\n').strip()
    if not text:
        return []
    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
    chunks = []
    current = ''
    for paragraph in paragraphs:
        candidate = f"{current}\n\n{paragraph}".strip() if current else paragraph
        if len(candidate) <= chunk_size:
            current = candidate
            continue
        if current:
            chunks.append(current)
        if len(paragraph) <= chunk_size:
            current = paragraph
            continue
        start = 0
        step = max(1, chunk_size - chunk_overlap)
        while start < len(paragraph):
            piece = paragraph[start:start + chunk_size].strip()
            if piece:
                chunks.append(piece)
            start += step
        current = ''
    if current:
        chunks.append(current)
    return chunks

def load_chunks(source_dir, chunk_size=900, chunk_overlap=150):
    docs = []
    for path in sorted(Path(source_dir).rglob('*.md')):
        text = path.read_text(encoding='utf-8')
        title = next((line.lstrip('#').strip() for line in text.splitlines() if line.strip().startswith('#')), path.stem)
        for chunk_id, content in enumerate(chunk_text(text, chunk_size=chunk_size, chunk_overlap=chunk_overlap)):
            docs.append({
                'chunk_id': chunk_id,
                'source': path.relative_to(source_dir).as_posix(),
                'title': title,
                'content': content,
            })
    return docs

def build_index():
    ensure_starter_docs()
    chunks = load_chunks(KNOWLEDGE_DIR, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    if not chunks:
        raise RuntimeError('No markdown files found in knowledge directory.')
    texts = [chunk['content'] for chunk in chunks]
    vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
    matrix = vectorizer.fit_transform(texts)
    with open(INDEX_PATH, 'wb') as f:
        pickle.dump({
            'embedding_model': EMBEDDING_MODEL,
            'vectorizer': vectorizer,
            'matrix': matrix,
            'chunks': chunks,
        }, f)
    manifest = {
        'built_at': datetime.utcnow().isoformat() + 'Z',
        'embedding_model': EMBEDDING_MODEL,
        'source_dir': KNOWLEDGE_DIR,
        'index_path': INDEX_PATH,
        'documents': len({chunk['source'] for chunk in chunks}),
        'chunks': len(chunks),
        'chunk_size': CHUNK_SIZE,
        'chunk_overlap': CHUNK_OVERLAP,
    }
    with open(MANIFEST_PATH, 'w', encoding='utf-8') as f:
        json.dump(manifest, f, indent=2)
    print('Saved index to', INDEX_PATH)
    print('Saved manifest to', MANIFEST_PATH)
    print(json.dumps(manifest, indent=2))
    return vectorizer, matrix, chunks

vectorizer, matrix, chunks = build_index()
print('Example files:', sorted(p.name for p in Path(KNOWLEDGE_DIR).glob('*.md')))


In [ ]:
def search(query, top_k=3):
    qv = vectorizer.transform([query])
    scores = (qv @ matrix.T).toarray().ravel()
    ranked = np.argsort(scores)[::-1]
    results = []
    for idx in ranked:
        score = float(scores[idx])
        if score <= 0:
            continue
        item = chunks[int(idx)]
        results.append({
            'score': round(score, 4),
            'source': item['source'],
            'title': item['title'],
            'preview': item['content'][:180],
        })
        if len(results) >= top_k:
            break
    return results

search('What are the main risks of Uniswap?')